# Kaggle batch runner — many experiments per GPU session

Runs a list of `train_from_memmap.py` configs back to back and collects
every artifact under `/kaggle/working/out/<tag>/`. Edit the `EXPERIMENTS`
list, Run All, download the whole Output when done.

Setup identical to the single-member notebook: private dataset with the
two zips attached as Input, GPU on, Internet on.

In [ ]:
DATASET = '/kaggle/input/js-regen-pool'   # <- your dataset mount

import shutil, os
shutil.copytree(f'{DATASET}/js_pool_998_1698_r9', '/kaggle/tmp/js_pool', dirs_exist_ok=True)
for d in ('src', 'scripts'):
    shutil.copytree(f'{DATASET}/{d}', f'/kaggle/working/project/{d}', dirs_exist_ok=True)
%pip install --quiet 'polars>=1.20' 'pyarrow>=15' 'tqdm>=4.66'
import torch; assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ---- subdecay wave (2026-07-20) ------------------------------------------
# The validated recipe (cheap-member proof: [96,96] 500-of-700 + hl175 =
# +0.01026 solo, best single member ever) applied to the TOP full-size
# models: each seed draws its own ~500 dates from the 600-date pool
# (998-1597) with half-life-175 recency decay. Different draws per seed =
# decorrelation by construction. --keep-epochs enables SETOL-picked
# stopping + {peak-1,peak,peak+1} epoch bagging post hoc.
# Contiguous-500d baselines to beat: gru_wide +0.00999, xsec_wide
# +0.00972, gru96 +0.00951 online.
EXPERIMENTS = [
    dict(tag='gw_sd42', model='gru_modelr', seed=42, extra='--hidden 128,128,128 --batch-size 2'),
    dict(tag='gw_sd1',  model='gru_modelr', seed=1,  extra='--hidden 128,128,128 --batch-size 2'),
    dict(tag='gw_sd2',  model='gru_modelr', seed=2,  extra='--hidden 128,128,128 --batch-size 2'),
    dict(tag='xw_sd42', model='gru_modelr_xsec', seed=42, extra='--hidden 128,128,128 --batch-size 2'),
    dict(tag='xw_sd1',  model='gru_modelr_xsec', seed=1,  extra='--hidden 128,128,128 --batch-size 2'),
    dict(tag='g96_sd42', model='gru_modelr', seed=42),
    dict(tag='g96_sd1',  model='gru_modelr', seed=1),
]

COMMON = ('--data /kaggle/tmp/js_pool '
          '--resample-mode subsample --resample-frac 0.833 --pool-lo 998 '
          '--decay-halflife 175 '
          '--valid-lo 1599 --valid-hi 1698 '
          '--device cuda --batch-size 4 --epochs 30 --patience 5 '
          '--watch --keep-epochs')


In [ ]:
import subprocess, json, time, os
os.environ['PYTHONPATH'] = '/kaggle/working/project/src'
results = []
for exp in EXPERIMENTS:
    out = f"/kaggle/working/out/{exp['tag']}"
    cmd = (f"python -u scripts/train_from_memmap.py {COMMON} "
           f"--model {exp['model']} --seed {exp.get('seed', 42)} "
           f"{exp.get('extra', '')} --tag {exp['tag']} --out {out}")
    print('=' * 70); print('RUN', exp['tag']); t0 = time.time()
    rc = subprocess.run(cmd, shell=True, cwd='/kaggle/working/project').returncode
    row = {'tag': exp['tag'], 'rc': rc, 'min': round((time.time() - t0) / 60, 1)}
    try:
        j = json.load(open(f"{out}/{exp['model']}_{exp['tag']}.json"))
        row.update(r2_static=j['r2_static'], r2_online=j['r2_online'])
    except Exception as e:
        row['error'] = str(e)
    results.append(row); print(row)

print('\n===== SESSION SUMMARY =====')
for r in results: print(r)
json.dump(results, open('/kaggle/working/out/session_summary.json', 'w'), indent=2)


Download the full `/kaggle/working/out` from the Output tab. Locally,
drop each member's `preds/*_online.npz` into
`artifacts/bench/regen_ensemble/preds/` for blend evaluation.